# 01 - Data Collection

## Objectives

* Download the cherry leaf image dataset from Kaggle (endpoint) and save it as raw data.
* Prepare the data for later steps: remove non-image files and split the images into train, validation and test sets.

## Inputs

* Kaggle API token: `kaggle.json` in the repository root (never committed - it is in `.gitignore`).
* Kaggle dataset: https://www.kaggle.com/codeinstitute/cherry-leaves

## Outputs

* `inputs/cherry_leaves_dataset/cherry-leaves/train`
* `inputs/cherry_leaves_dataset/cherry-leaves/validation`
* `inputs/cherry_leaves_dataset/cherry-leaves/test`

## Additional comments

* This notebook covers the **Data Understanding** and **Data Preparation** steps of CRISP-DM.
* The image data is not pushed to GitHub (too large); it is recreated by running this notebook.

---

# Change working directory

The notebooks live in `jupyter_notebooks/`, but all paths in this project are relative to the repository root, so we move one folder up.

In [ ]:
import os

current_dir = os.getcwd()
if os.path.basename(current_dir) == 'jupyter_notebooks':
    os.chdir(os.path.dirname(current_dir))
print('Working directory:', os.getcwd())

---

# Download the dataset from Kaggle

We point the Kaggle library to the `kaggle.json` file in the repository root, then download and unzip the dataset. If the data is already there, the download is skipped.

In [ ]:
import os

DATASET = 'codeinstitute/cherry-leaves'
DATA_FOLDER = os.path.join('inputs', 'cherry_leaves_dataset')

# The Kaggle library looks for kaggle.json in this folder
os.environ['KAGGLE_CONFIG_DIR'] = os.getcwd()

In [ ]:
if os.path.isdir(DATA_FOLDER) and os.listdir(DATA_FOLDER):
    print('Dataset already downloaded - skipping.')
else:
    # Imported here because it authenticates as soon as it is imported
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(DATASET, path=DATA_FOLDER, unzip=True)
    print('Dataset downloaded to', DATA_FOLDER)

Find the folder that holds the two class sub-folders (`healthy` and `powdery_mildew`).

In [ ]:
def find_class_root(base_folder):
    """Return the folder that directly contains the 'healthy' sub-folder."""
    for root, dirs, _ in os.walk(base_folder):
        if 'healthy' in dirs and 'powdery_mildew' in dirs:
            return root
    raise FileNotFoundError('Could not find the healthy / powdery_mildew folders')


my_data_dir = find_class_root(DATA_FOLDER)
print('Images are in:', my_data_dir)
print('Classes:', sorted(os.listdir(my_data_dir)))

---

# Data cleaning: remove non-image files

Only `.png`, `.jpg` and `.jpeg` files are useful for training. Any other file is deleted.

In [ ]:
IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg')


def remove_non_image_files(my_data_dir):
    """Delete every file that is not an image and report what was kept."""
    for label in sorted(os.listdir(my_data_dir)):
        folder = os.path.join(my_data_dir, label)
        if not os.path.isdir(folder):
            continue
        kept, removed = 0, 0
        for file_name in os.listdir(folder):
            if file_name.lower().endswith(IMAGE_EXTENSIONS):
                kept += 1
            else:
                os.remove(os.path.join(folder, file_name))
                removed += 1
        print(f'{label}: {kept} image files kept, {removed} non-image files removed')


remove_non_image_files(my_data_dir)

---

# Split into train, validation and test sets

* **Train (70%)** - the images the model learns from.
* **Validation (10%)** - used during training to check progress and stop early.
* **Test (20%)** - kept aside and only used at the end to measure real performance.

The split is random but uses a fixed seed so it can be reproduced.

In [ ]:
import random
import shutil


def split_train_validation_test_images(my_data_dir, train_ratio=0.7,
                                       validation_ratio=0.1, test_ratio=0.2,
                                       seed=42):
    """Move each class's images into train/validation/test sub-folders."""
    if abs(train_ratio + validation_ratio + test_ratio - 1.0) > 1e-6:
        raise ValueError('The three ratios must add up to 1.0')

    sets = ['train', 'validation', 'test']
    if all(os.path.isdir(os.path.join(my_data_dir, s)) for s in sets):
        print('Data is already split - skipping.')
        return

    labels = [d for d in os.listdir(my_data_dir) if d not in sets]
    random.seed(seed)
    for label in labels:
        label_dir = os.path.join(my_data_dir, label)
        files = sorted(os.listdir(label_dir))
        random.shuffle(files)

        n_train = int(len(files) * train_ratio)
        n_val = int(len(files) * validation_ratio)
        groups = {'train': files[:n_train],
                  'validation': files[n_train:n_train + n_val],
                  'test': files[n_train + n_val:]}

        for set_name, set_files in groups.items():
            target = os.path.join(my_data_dir, set_name, label)
            os.makedirs(target, exist_ok=True)
            for file_name in set_files:
                shutil.move(os.path.join(label_dir, file_name),
                            os.path.join(target, file_name))
        os.rmdir(label_dir)  # now empty


split_train_validation_test_images(my_data_dir)

In [ ]:
# Check the result: number of images per set and label
for set_name in ['train', 'validation', 'test']:
    for label in sorted(os.listdir(os.path.join(my_data_dir, set_name))):
        n = len(os.listdir(os.path.join(my_data_dir, set_name, label)))
        print(f'{set_name:<11} {label:<15} {n} images')

---

# Conclusions and next steps

* The dataset was downloaded from the Kaggle endpoint, cleaned of non-image files and split 70/10/20 into train, validation and test sets.
* Both classes have a similar number of images, so the dataset is balanced.
* **Next:** `02_DataVisualization.ipynb` - study the images to answer business requirement 1.